## 1. Setup - Mount Google Drive

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("✓ Google Drive mounted successfully!")
print("\nĐường dẫn file của bạn sẽ là:")
print("/content/drive/MyDrive/<tên_folder>/cleaned_real_estate.csv")

## 2. Configuration - Đường dẫn file

In [ ]:
# ⚠️ QUAN TRỌNG: Thay đổi đường dẫn này theo vị trí file của bạn
DATA_PATH = "/content/drive/MyDrive/DO-AN-TOT-NGHIEP/cleaned_real_estate.csv"

# Thư mục lưu models (trên Google Drive)
MODELS_DIR = "/content/drive/MyDrive/DO-AN-TOT-NGHIEP/models"
LOGS_DIR = "/content/drive/MyDrive/DO-AN-TOT-NGHIEP/logs"

# Tạo thư mục nếu chưa có
import os
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)

print(f"✓ Data path: {DATA_PATH}")
print(f"✓ Models will be saved to: {MODELS_DIR}")
print(f"✓ Logs will be saved to: {LOGS_DIR}")

## 3. Install Libraries

In [ ]:
!pip install -q xgboost lightgbm

print("✓ All libraries installed!")

## 4. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import joblib
import time
from pathlib import Path
from datetime import datetime

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# Configuration
RANDOM_STATE = 42
N_JOBS = -1
CV_FOLDS = 5
TEST_SIZE = 0.2

print("✓ All libraries imported!")

## 5. Load and Explore Data

In [ ]:
print("Loading data...")
start_time = time.time()

# Load data - sử dụng toàn bộ dataset hoặc sample để test nhanh
USE_SAMPLE = True  # Đổi thành False để train trên toàn bộ data
SAMPLE_FRAC = 0.1  # 10% data cho test nhanh

if USE_SAMPLE:
    df = pd.read_csv(DATA_PATH)
    df = df.sample(frac=SAMPLE_FRAC, random_state=RANDOM_STATE)
    print(f"✓ Loaded {len(df):,} rows (sample {SAMPLE_FRAC*100:.0f}%)")
else:
    df = pd.read_csv(DATA_PATH)
    print(f"✓ Loaded {len(df):,} rows (full dataset)")

load_time = time.time() - start_time
print(f"  Load time: {load_time:.2f}s")
print(f"  Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Display info
print("\nDataset Info:")
print(f"  Shape: {df.shape}")
print(f"  Columns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Data analysis
print("=== DATA ANALYSIS ===")
print(f"\n1. Missing values:")
print(df.isnull().sum())

print(f"\n2. Target (price) statistics:")
print(df['price'].describe())

print(f"\n3. Year range: {df['year'].min()} - {df['year'].max()}")
print(f"   Total years: {df['year'].nunique()}")

print(f"\n4. Countries: {df['country'].unique()}")
print(f"   Total cities: {df['city'].nunique()}")

print(f"\n5. Property types:")
print(df['property_type'].value_counts().head(10))

## 6. Feature Engineering

In [ ]:
print("Creating advanced features...\n")

# 1. TEMPORAL FEATURES
print("1. Temporal features...")
df['years_since_1990'] = df['year'] - 1990
df['year_squared'] = df['year'] ** 2
df['quarter'] = (df['month'] - 1) // 3 + 1
df['is_peak_season'] = df['month'].isin([3, 4, 5, 6]).astype(int)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

# 2. INTERACTION FEATURES
print("2. Interaction features...")
df['price_per_m2_x_area'] = df['price_per_m2'] * df['area_m2']
df['price_per_m2_x_year'] = df['price_per_m2'] * df['years_since_1990']
df['area_x_year'] = df['area_m2'] * df['years_since_1990']

# 3. AREA FEATURES
print("3. Area features...")
df['log_area'] = np.log1p(df['area_m2'])
df['area_squared'] = df['area_m2'] ** 2

# 4. PRICE FEATURES
print("4. Price features...")
df['log_price_per_m2'] = np.log1p(df['price_per_m2'])

print(f"\n✓ Feature engineering complete!")
print(f"  Total features: {len(df.columns)}")
print(f"  New features: {len(df.columns) - 9}")

df.head()

## 7. Build Preprocessing Pipeline

In [ ]:
print("Building preprocessing pipeline...\n")

# Prepare features
target = 'price'
X = df.drop(columns=[target, 'date'])  # Remove target and date
y = df[target]

# Identify feature types
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Feature types:")
print(f"  Numeric: {len(numeric_features)} features")
print(f"  Categorical: {len(categorical_features)} features")
print(f"    {categorical_features}")

# Build transformers
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False, max_categories=50))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='drop'
)

print("\n✓ Preprocessor ready!")

## 8. Split Data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print("Data split:")
print(f"  Training: {len(X_train):,} samples")
print(f"  Test: {len(X_test):,} samples")
print(f"\nTarget statistics:")
print(f"  Train mean: {y_train.mean():,.2f}")
print(f"  Test mean: {y_test.mean():,.2f}")

## 9. Define Models

In [ ]:
models = {
    'LinearRegression': LinearRegression(n_jobs=N_JOBS),
    
    'Ridge': Ridge(alpha=10.0, random_state=RANDOM_STATE),
    
    'Lasso': Lasso(alpha=10.0, random_state=RANDOM_STATE, max_iter=2000),
    
    'RandomForest': RandomForestRegressor(
        n_estimators=100,
        max_depth=20,
        min_samples_split=10,
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
        verbose=1
    ),
    
    'ExtraTrees': ExtraTreesRegressor(
        n_estimators=100,
        max_depth=20,
        min_samples_split=10,
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
        verbose=1
    ),
    
    'GradientBoosting': GradientBoostingRegressor(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        random_state=RANDOM_STATE,
        verbose=1
    ),
    
    'XGBoost': XGBRegressor(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
        verbosity=1
    ),
    
    'LightGBM': LGBMRegressor(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
        verbose=1
    )
}

print(f"✓ Defined {len(models)} models")
for name in models.keys():
    print(f"  - {name}")

## 10. Train Models

In [ ]:
print("="*80)
print("TRAINING MODELS")
print("="*80)

all_results = []
trained_models = {}

for name, model in models.items():
    print(f"\n{'='*80}")
    print(f"Training: {name}")
    print(f"{'='*80}")
    
    # Create pipeline
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    # Train
    start_time = time.time()
    pipeline.fit(X_train, y_train)
    train_time = time.time() - start_time
    
    # Evaluate
    train_pred = pipeline.predict(X_train)
    test_pred = pipeline.predict(X_test)
    
    train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
    train_r2 = r2_score(y_train, train_pred)
    test_r2 = r2_score(y_test, test_pred)
    test_mae = mean_absolute_error(y_test, test_pred)
    
    # Store results
    results = {
        'model_name': name,
        'train_rmse': train_rmse,
        'test_rmse': test_rmse,
        'train_r2': train_r2,
        'test_r2': test_r2,
        'test_mae': test_mae,
        'train_time': train_time
    }
    all_results.append(results)
    trained_models[name] = pipeline
    
    # Display
    print(f"\nResults:")
    print(f"  Train RMSE: {train_rmse:,.2f}")
    print(f"  Test RMSE:  {test_rmse:,.2f}")
    print(f"  Train R²:   {train_r2:.4f}")
    print(f"  Test R²:    {test_r2:.4f}")
    print(f"  Test MAE:   {test_mae:,.2f}")
    print(f"  Time:       {train_time:.2f}s")

print(f"\n{'='*80}")
print("ALL MODELS TRAINED!")
print(f"{'='*80}")

## 11. Results Summary

In [ ]:
# Create results DataFrame
results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values('test_rmse')

print("\n" + "="*80)
print("MODEL RANKINGS (by Test RMSE)")
print("="*80)
print(results_df.to_string(index=False))

# Best model
best_model_name = results_df.iloc[0]['model_name']
best_results = results_df.iloc[0]

print(f"\n{'='*80}")
print("BEST MODEL")
print(f"{'='*80}")
print(f"  Model: {best_model_name}")
print(f"  Test RMSE: {best_results['test_rmse']:,.2f}")
print(f"  Test R²: {best_results['test_r2']:.4f}")
print(f"  Test MAE: {best_results['test_mae']:,.2f}")
print(f"  Train time: {best_results['train_time']:.2f}s")

## 12. Save Models to Google Drive

In [ ]:
print("Saving models to Google Drive...\n")

# Save best model with metadata
best_model = trained_models[best_model_name]
feature_names = X_train.columns.tolist()

model_data = {
    'pipeline': best_model,
    'feature_names': feature_names,
    'model_name': best_model_name,
    'target': target,
    'test_rmse': best_results['test_rmse'],
    'test_r2': best_results['test_r2'],
    'test_mae': best_results['test_mae'],
    'train_time': best_results['train_time'],
    'training_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'training_samples': len(X_train),
    'test_samples': len(X_test),
    'total_features': len(feature_names),
    'random_state': RANDOM_STATE,
    'trained_on': 'Google Colab'
}

best_model_path = f"{MODELS_DIR}/best_colab.joblib"
joblib.dump(model_data, best_model_path)
print(f"✓ Best model saved: {best_model_path}")

# Save all models
print("\nSaving all models...")
for name, model in trained_models.items():
    model_path = f"{MODELS_DIR}/{name.lower().replace(' ', '_')}_colab.joblib"
    joblib.dump(model, model_path)
    print(f"  ✓ {name}")

# Save results log
log_path = f"{LOGS_DIR}/training_log_colab.csv"
results_df['timestamp'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
results_df['trained_on'] = 'Google Colab'
results_df.to_csv(log_path, index=False)
print(f"\n✓ Training log saved: {log_path}")

print("\n" + "="*80)
print("ALL FILES SAVED TO GOOGLE DRIVE!")
print("="*80)
print(f"Models directory: {MODELS_DIR}")
print(f"Logs directory: {LOGS_DIR}")
print("\nBạn có thể download các file này về máy từ Google Drive.")

## 13. Test Predictions

In [ ]:
# Test với một số samples
test_samples = X_test.head(10)
predictions = best_model.predict(test_samples)
actual = y_test.head(10).values

comparison = pd.DataFrame({
    'Actual': actual,
    'Predicted': predictions,
    'Difference': actual - predictions,
    'Error_%': ((actual - predictions) / actual * 100).round(2)
})

print("Sample Predictions:")
print(comparison.to_string())

print(f"\nAverage Error: {comparison['Error_%'].abs().mean():.2f}%")

## 14. Download Models (Optional)

Nếu muốn download trực tiếp từ Colab về máy (không qua Drive):

In [ ]:
from google.colab import files

# Download best model
print("Downloading best model...")
files.download(best_model_path)

# Download training log
print("Downloading training log...")
files.download(log_path)

print("\n✓ Files downloaded!")
print("Giờ bạn có thể copy các file này vào thư mục models/ trên ổ D")

## 🎯 Next Steps

### Để sử dụng model đã train:

1. **Download từ Google Drive:**
   - File `best_colab.joblib` từ folder models
   - Copy vào: `D:\Do An Tot Nghiep...\models\`

2. **Test model trên máy local:**
   ```bash
   python predict.py --model models/best_colab.joblib --input Data/cleaned_real_estate.csv --sample 0.01
   ```

3. **Cập nhật web app:**
   - Web app sẽ tự động load model mới nếu bạn đặt vào thư mục models/
   - Hoặc restart Flask server: `python app.py`

### Để train trên toàn bộ dataset (978K rows):
   - Đổi `USE_SAMPLE = False` ở cell "5. Load and Explore Data"
   - Runtime sẽ mất khoảng 15-30 phút
   - Cần GPU runtime để nhanh hơn: Runtime > Change runtime type > GPU

### Để train với nhiều năm hơn:
   - Data đã có 35 năm (1990-2025) nên model đã được train cho dự đoán dài hạn
   - Có thể filter data theo năm cụ thể nếu cần:
     ```python
     df = df[df['year'] >= 2015]  # Chỉ dùng data từ 2015 trở đi
     ```